<a href="https://colab.research.google.com/github/liuyunxiao1020-code/5m-data-3.10-nlp-advanced/blob/main/assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment

## Instructions

Use the following code as a starting point to load the rotten tomatoes dataset:

```python
from datasets import load_dataset

# Load the Rotten Tomatoes dataset
dataset = load_dataset("rotten_tomatoes")

# Print the dataset information
print(dataset)

# Example: Accessing the training split
train_dataset = dataset["train"]

# Print the first example in the training set
print(train_dataset[0])
```

**Model Application:**

- Load a pre-trained sentiment analysis model from Hugging Face Transformers.
- Apply the model to a subset of the chosen dataset (e.g., the first 1000 samples from the training set).
- Evaluate the model's performance. You can start with qualitative analysis (inspecting predictions) and then explore quantitative metrics.

## Submission

- Submit the URL of the GitHub Repository that contains your work to NTU black board.
- Should you reference the work of your classmate(s) or online resources, give them credit by adding either the name of your classmate or URL.


In [1]:
!pip install -q datasets transformers

In [5]:
import random
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

In [6]:
# 1. Load the dataset

print("Loading Rotten Tomatoes dataset...")
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(dataset)

train_dataset = dataset["train"]
print("\nFirst example in the training set:")
print(train_dataset[0])

Loading Rotten Tomatoes dataset...


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

First example in the training set:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [7]:
# 2. Load a pre-trained sentiment analysis model
# distilbert-base-uncased-finetuned-sst-2-english is a small, fast, widely-used binary sentiment classifier well suited to short movie-review-style text.
print("\nLoading pre-trained sentiment analysis pipeline...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)


Loading pre-trained sentiment analysis pipeline...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [12]:
# 3. Apply the model to a subset of the data (first 1000 training samples)
N = 1000
subset = train_dataset.shuffle(seed=42).select(range(N))
# Force plain python lists -- some `datasets` versions return lazy column objects or non-str elements here, which the pipeline rejects.
texts = [str(t) for t in subset["text"]]
true_labels = list(subset["label"])  # 0 = negative, 1 = positive

print(f"\nRunning inference on {N} samples...")
# Batch the predictions for speed
raw_predictions = sentiment_pipeline(texts, batch_size=32, truncation=True)



Running inference on 1000 samples...


In [13]:
# Map model output labels ("NEGATIVE"/"POSITIVE") to the dataset's 0/1 scheme
label_map = {"NEGATIVE": 0, "POSITIVE": 1}
predicted_labels = [label_map[pred["label"]] for pred in raw_predictions]
confidences = [pred["score"] for pred in raw_predictions]

results_df = pd.DataFrame(
    {
        "text": texts,
        "true_label": true_labels,
        "predicted_label": predicted_labels,
        "confidence": confidences,
    }
)
results_df["correct"] = results_df["true_label"] == results_df["predicted_label"]

In [14]:
results_df.describe()

,true_label,predicted_label,confidence
count,1000.000000,1000.000000,1000.000000
mean,0.488000,0.515000,0.987641
std,0.500106,0.500025,0.047254
min,0.000000,0.000000,0.511960
25%,0.000000,0.000000,0.997515
50%,0.000000,1.000000,0.999546
75%,1.000000,1.000000,0.999792
max,1.000000,1.000000,0.999892


In [15]:
# 4. Qualitative analysis: inspect a handful of predictions
print("\n" + "=" * 70)
print("QUALITATIVE ANALYSIS")
print("=" * 70)

label_name = {0: "NEGATIVE", 1: "POSITIVE"}

print("\n--- 5 random correct predictions ---")
correct_samples = results_df[results_df["correct"]].sample(
    n=min(5, results_df["correct"].sum()), random_state=42
)
for _, row in correct_samples.iterrows():
    print(f"\nText: {row['text']}")
    print(
        f"True: {label_name[row['true_label']]} | "
        f"Predicted: {label_name[row['predicted_label']]} "
        f"(confidence: {row['confidence']:.3f})"
    )

print("\n--- Up to 5 random misclassified predictions ---")
incorrect_samples = results_df[~results_df["correct"]]
if len(incorrect_samples) > 0:
    incorrect_samples = incorrect_samples.sample(
        n=min(5, len(incorrect_samples)), random_state=42
    )
    for _, row in incorrect_samples.iterrows():
        print(f"\nText: {row['text']}")
        print(
            f"True: {label_name[row['true_label']]} | "
            f"Predicted: {label_name[row['predicted_label']]} "
            f"(confidence: {row['confidence']:.3f})"
        )
else:
    print("No misclassifications in this subset!")


QUALITATIVE ANALYSIS

--- 5 random correct predictions ---

Text: it deserves to be seen by anyone with even a passing interest in the events shaping the world beyond their own horizons .
True: POSITIVE | Predicted: POSITIVE (confidence: 0.957)

Text: manages to please its intended audience -- children -- without placing their parents in a coma-like state .
True: POSITIVE | Predicted: POSITIVE (confidence: 1.000)

Text: the movie fails to live up to the sum of its parts .
True: NEGATIVE | Predicted: NEGATIVE (confidence: 1.000)

Text: if anything , the film is doing something of a public service -- shedding light on a group of extremely talented musicians who might otherwise go unnoticed and underappreciated by music fans .
True: POSITIVE | Predicted: POSITIVE (confidence: 0.999)

Text: singer/composer bryan adams contributes a slew of songs  a few potential hits , a few more simply intrusive to the story  but the whole package certainly captures the intended , er , spirit of the pi

In [16]:
# 5. Quantitative evaluation
print("\n" + "=" * 70)
print("QUANTITATIVE EVALUATION")
print("=" * 70)

accuracy = accuracy_score(true_labels, predicted_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels, predicted_labels, average="binary"
)

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nFull classification report:")
print(
    classification_report(
        true_labels, predicted_labels, target_names=["NEGATIVE", "POSITIVE"]
    )
)

cm = confusion_matrix(true_labels, predicted_labels)
print("Confusion Matrix:")
print("                 Predicted NEG   Predicted POS")
print(f"Actual NEG       {cm[0][0]:<15}{cm[0][1]}")
print(f"Actual POS       {cm[1][0]:<15}{cm[1][1]}")


QUANTITATIVE EVALUATION

Accuracy:  0.8810
Precision: 0.8583
Recall:    0.9057
F1 Score:  0.8814

Full classification report:
              precision    recall  f1-score   support

    NEGATIVE       0.91      0.86      0.88       512
    POSITIVE       0.86      0.91      0.88       488

    accuracy                           0.88      1000
   macro avg       0.88      0.88      0.88      1000
weighted avg       0.88      0.88      0.88      1000

Confusion Matrix:
                 Predicted NEG   Predicted POS
Actual NEG       439            73
Actual POS       46             442


In [17]:
# Confidence analysis: is the model more/less confident when it's wrong?
print("\nAverage confidence when correct:  ", round(results_df.loc[results_df.correct, "confidence"].mean(), 4))
if (~results_df.correct).any():
    print("Average confidence when incorrect:", round(results_df.loc[~results_df.correct, "confidence"].mean(), 4))

# Save full results for further inspection
results_df.to_csv("sentiment_predictions.csv", index=False)
print("\nSaved detailed predictions to 'sentiment_predictions.csv'")


Average confidence when correct:   0.9935
Average confidence when incorrect: 0.9441

Saved detailed predictions to 'sentiment_predictions.csv'
